In [1]:
pip install scikit-optimize

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.regularizers import l2

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

# scikit-optimize imports for Bayesian tuning
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

In [3]:
# 2. NumPy
np.random.seed(42)

# 3. TensorFlow
tf.random.set_seed(42)

In [4]:
# Load data
df = pd.read_csv("./processed_plays.csv")

# Train/Validation/Test split (60/20/20)
train_val, test = train_test_split(df, test_size=0.20, random_state=42)
train, val      = train_test_split(train_val, test_size=0.20, random_state=42)

X_train, y_train = train.drop('yardsGained', axis=1), train['yardsGained']
X_val,   y_val   = val.drop('yardsGained', axis=1),   val['yardsGained']
X_test,  y_test  = test.drop('yardsGained', axis=1),  test['yardsGained']

In [5]:
df.head()

,down,yardsToGo,absoluteYardlineNumber,quarter,playAction,qbSneak,pff_runPassOption,yardsGained,secondsRemainingInQuarter,offFormation_EMPTY,...,passCoverage_Cover_3_Seam,passCoverage_Cover_6_Right,passCoverage_Goal_Line,passCoverage_Miscellaneous,passCoverage_Prevent,passCoverage_Quarters,passCoverage_Red_Zone,manZone_Man,manZone_Other,manZone_Zone
0,1,10,21,3,0,0,0,9,114,1,...,0,0,0,0,0,0,0,0,0,1
1,1,10,8,4,0,0,0,4,133,1,...,0,0,0,0,0,1,0,0,0,1
2,3,12,20,4,0,0,0,6,120,0,...,0,0,0,0,0,1,0,0,0,1
3,2,10,23,1,0,0,0,4,568,0,...,0,0,0,0,0,1,0,0,0,1
4,2,8,27,3,1,0,0,-1,136,0,...,0,0,0,0,0,0,0,1,0,0


In [6]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Define model architecture
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(40, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(1)
])


In [7]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',  # Mean Squared Error
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'),
             tf.keras.metrics.MeanAbsoluteError(name='mae')]
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on train and validation sets
train_metrics = model.evaluate(X_train_scaled, y_train, verbose=0)
val_metrics   = model.evaluate(X_val_scaled, y_val, verbose=0)

print(f"\nNeural Net on train data → RMSE: {train_metrics[1]:.3f}  MAE: {train_metrics[2]:.3f}")
print(f"Neural Net on validation data → RMSE: {val_metrics[1]:.3f}  MAE: {val_metrics[2]:.3f}")

Epoch 1/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 85.9507 - mae: 5.9955 - rmse: 9.2629 - val_loss: 68.0430 - val_mae: 5.5033 - val_rmse: 8.2413
Epoch 2/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 77.7617 - mae: 5.8429 - rmse: 8.8076 - val_loss: 67.6559 - val_mae: 5.5268 - val_rmse: 8.2178
Epoch 3/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 76.2363 - mae: 5.8028 - rmse: 8.7203 - val_loss: 67.5330 - val_mae: 5.5040 - val_rmse: 8.2103
Epoch 4/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 75.5519 - mae: 5.7629 - rmse: 8.6800 - val_loss: 67.3800 - val_mae: 5.5070 - val_rmse: 8.2009
Epoch 5/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 74.6669 - mae: 5.7287 - rmse: 8.6289 - val_loss: 67.4188 - val_mae: 5.5052 - val_rmse: 8.2033
Epoch 6/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 75.0623 - mae: 5.7532 - rmse: 8.6514 - val_loss: 67.4534 - val_mae: 5.5261 - val_rmse: 8.2053
Epoch 7/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 74.7277 -

In [8]:
model.predict(X_val_scaled[0].reshape(1, -1))[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


5.283851

In [9]:
X_val_scaled[0]

array([-0.97476246,  0.39093328,  1.38895517, -1.37883379, -0.45855187,
       -0.08842771, -0.33692293,  1.7080902 , -0.30469828, -0.26092011,
       -0.08899002, -0.20247927, -1.11655282,  1.7608761 , -0.07301886,
       -0.86716475,  1.02567225, -0.11193674, -0.15100534, -0.13715424,
       -0.08497816, -0.24374391, -0.04653337, -0.08899002, -1.03722754,
       -0.06882246, -0.36913021,  2.69653929, -1.20218341, -0.28962619,
       -0.3159202 , -0.17513172, -0.1146051 , -0.05251223, -0.35314309,
       -0.29810653, -0.97900641, -0.03710618, -0.42019728, -0.24622145,
       -0.18473098, -0.08842771, -0.0861429 ,  7.86459483, -0.10593661,
       -0.07369549, -0.21327732, -0.193096  , -0.5169197 , -0.05614899,
       -0.36205246,  1.48966298, -0.04436346, -0.04758153, -0.026229  ,
       -0.20457271, -0.21578189, -0.09065669, -0.03435027, -0.05788276,
       -0.38358617, -0.18473098, -0.59436482, -0.23198941,  0.67360279])

In [10]:
def evaluate(name, actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    print(f"{name} → RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")

evaluate('Train', y_train, model.predict(X_train_scaled).flatten())
evaluate('Validation', y_val,   model.predict(X_val_scaled).flatten())
evaluate('Test', y_test,  model.predict(X_test_scaled).flatten())


319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 516us/step
Train → RMSE: 8.595, MAE: 5.626, R²: 0.064
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 633us/step
Validation → RMSE: 8.192, MAE: 5.498, R²: 0.021
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 596us/step
Test → RMSE: 9.052, MAE: 5.783, R²: 0.030


In [14]:
!pip install scikeras

In [16]:
import scikeras
from scikeras.wrappers import KerasRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.pipeline import Pipeline

def build_model(n1=64, n2=40, dropout_rate=0.2, learning_rate=0.001, l2_reg=0.001):
    model = Sequential([
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(n1, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(n2, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'),
                 tf.keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return model

# Wrap the model
regressor = KerasRegressor(
    model=build_model,
    verbose=0,
    epochs=50,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

# Search space
param_space = {
    "model__n1": Integer(32, 256),
    "model__n2": Integer(16, 128),
    "model__dropout_rate": Real(0.0, 0.5),
    "model__learning_rate": Real(1e-4, 1e-2, prior='log-uniform'),
    "model__l2_reg": Real(1e-5, 1e-2, prior='log-uniform'),
}

# Build search
opt = BayesSearchCV(
    estimator=regressor,
    search_spaces=param_space,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Run tuning
opt.fit(X_train_scaled, y_train)

# Results
print(f"\n✅ Best Parameters:\n{opt.best_params_}")
print(f"Best RMSE (CV): {np.sqrt(-opt.best_score_):.3f}")


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

C:\Users\theod\anaconda3\Lib\site-packages\keras\src\callbacks\early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae,rmse
  current = self.get_monitor_value(logs)



✅ Best Parameters:
OrderedDict({'model__dropout_rate': 0.36587228895094365, 'model__l2_reg': 0.006040259819922876, 'model__learning_rate': 0.00014977042135838465, 'model__n1': 32, 'model__n2': 107})
Best RMSE (CV): 8.825


In [17]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Define model architecture
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(107, activation='relu', kernel_regularizer=l2(0.006)),
    Dropout(0.36),
    Dense(32, activation='relu', kernel_regularizer=l2(0.006)),
    Dropout(0.36),
    Dense(1)
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.00015),
    loss='mse',  # Mean Squared Error
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'),
             tf.keras.metrics.MeanAbsoluteError(name='mae')]
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on train and validation sets
train_metrics = model.evaluate(X_train_scaled, y_train, verbose=0)
val_metrics   = model.evaluate(X_val_scaled, y_val, verbose=0)

print(f"\nNeural Net on train data → RMSE: {train_metrics[1]:.3f}  MAE: {train_metrics[2]:.3f}")
print(f"Neural Net on validation data → RMSE: {val_metrics[1]:.3f}  MAE: {val_metrics[2]:.3f}")

Epoch 1/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 112.2371 - mae: 6.8264 - rmse: 10.5559 - val_loss: 79.2364 - val_mae: 5.5839 - val_rmse: 8.8571
Epoch 2/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 87.9146 - mae: 5.8862 - rmse: 9.3312 - val_loss: 71.5757 - val_mae: 5.5423 - val_rmse: 8.4121
Epoch 3/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 83.3922 - mae: 5.9505 - rmse: 9.0847 - val_loss: 69.8246 - val_mae: 5.5461 - val_rmse: 8.3070
Epoch 4/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 80.8587 - mae: 5.8832 - rmse: 8.9424 - val_loss: 69.1037 - val_mae: 5.5463 - val_rmse: 8.2635
Epoch 5/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 79.3736 - mae: 5.8462 - rmse: 8.8589 - val_loss: 68.7341 - val_mae: 5.5316 - val_rmse: 8.2414
Epoch 6/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 79.8958 - mae: 5.8442 - rmse: 8.8887 - val_loss: 68.5250 - val_mae: 5.5202 - val_rmse: 8.2289
Epoch 7/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 79.6479

In [18]:
def evaluate(name, actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    print(f"{name} → RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")

evaluate('Train', y_train, model.predict(X_train_scaled).flatten())
evaluate('Validation', y_val,   model.predict(X_val_scaled).flatten())
evaluate('Test', y_test,  model.predict(X_test_scaled).flatten())

319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 765us/step
Train → RMSE: 8.654, MAE: 5.705, R²: 0.051
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 881us/step
Validation → RMSE: 8.188, MAE: 5.531, R²: 0.022
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 789us/step
Test → RMSE: 9.038, MAE: 5.808, R²: 0.033
